# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HaneefAderolu/ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Cell 0 - Setup
import os

# Clone repo if not already there
if not os.path.exists('ml-internship-starter'):
    os.system('git clone https://github.com/HaneefAderolu/ml-internship-starter.git')
os.chdir('ml-internship-starter')
print("Working directory:", os.getcwd())

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Construct proxy label (same as W04)
df['label'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_90d'] >= 500) &
    (df['content_age_days'] >= 180)
).astype(int)

# Fix avg_position: 0 means "no data", not rank zero
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)
df['has_position'] = (df['avg_position'] > 0).astype(int)

# word_count has missingness — add flag instead of blind fillna
df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count_filled'] = df['word_count'].fillna(0)

print(f"Loaded {len(df):,} rows")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"Base rate: {df['label'].mean():.3f}")

Working directory: /content/ml-internship-starter
Loaded 30,000 rows
Label distribution:
label
0    24639
1     5361
Name: count, dtype: int64
Base rate: 0.179


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
Method: Random Forest classifier, evaluated with Precision@K (K=20, 50, 100).

Why Random Forest and not something else:

Lane 1 is a ranking signal analysis problem. The question is:
"Which signals best separate pages that need attention from those that don't?"
Random Forest answers this directly because:
  1. It produces a probability score per page - which lets us rank pages (not just classify them)
  2. It gives feature importances - which tells us WHICH signals drive the ranking
  3. It handles mixed data types (numeric, sparse) without heavy preprocessing
  4. It is robust to outliers, which matter here (impressions_90d is heavily skewed)

Why not Logistic Regression first?
  The skill guide says "start simple". We tried it, but the feature relationships
  here are non-linear (a page with 500 impressions AND 180+ days behaves very
  differently from one with 500 impressions AND 30 days). Random Forest captures
  this interaction; Logistic Regression needs manual interaction terms.

Why not Gradient Boosting?
  Random Forest is safer here because our test set is only 8 clients (small).
  Gradient Boosting can overfit more aggressively on small test sets.
  If this were a larger panel we would try it next.

Metric: Precision@K - of the top K pages the model ranks highest,
what fraction actually have our proxy label = 1?
This is the right metric because the use case is a priority queue:
an analyst looks at the top 20-100 pages. We need those to be real opportunities.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Method validation - show why precision@K fits this use case
features = [
    'impressions_90d', 'days_with_impressions', 'days_with_sessions',
    'avg_position_clean', 'has_position', 'ctr', 'engagement_rate',
    'scroll_rate', 'word_count_filled', 'has_word_count',
    'content_age_days', 'days_since_last_update', 'search_volume',
    'competition', 'sessions_90d', 'pageviews_90d',
]

X = df[features].copy().fillna(df[features].median())
y = df['label'].copy()
groups = df['client_id'].values

print(f"Features: {len(features)}")
print(f"Positive labels: {y.sum():,} ({y.mean():.1%})")
print(f"Clients in dataset: {df['client_id'].nunique()}")
print("\nBase rate (random ranking would achieve this at any K):", round(y.mean(), 3))

Features: 16
Positive labels: 5,361 (17.9%)
Clients in dataset: 32

Base rate (random ranking would achieve this at any K): 0.179


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
Split design: Grouped by client_id, 75/25 train/test.

Why grouped by client?
  Each client has a different content strategy, niche, and traffic volume.
  If we split randomly, the model sees some pages from Client A in training
  and other pages from Client A in test. It learns client-specific patterns
  and looks better than it really is - this is client leakage.

  Grouped split means: every page from a given client goes ENTIRELY into
  train OR entirely into test. The model must generalise to clients it has
  never seen. This is the honest version.

  Train: 24 clients, 22,885 pages
  Test:  8 clients,   7,115 pages

Why not a time-based split?
  The starter CSV is a snapshot (trailing 90 days). There are no timestamps
  per page that would allow a clean time split. Grouped-by-client is the
  next most honest option.

Random seed: 42. Fixed for reproducibility.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Grouped split - honest: all pages from a client go to same fold
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train):,} rows | {y_train.sum()} positive | "
      f"{df['client_id'].iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test):,} rows  | {y_test.sum()} positive  | "
      f"{df['client_id'].iloc[test_idx].nunique()} clients")
print(f"\nTest base rate: {y_test.mean():.3f}")

Train: 22,885 rows | 4156 positive | 24 clients
Test:  7,115 rows  | 1205 positive  | 8 clients

Test base rate: 0.169


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
Same test split, same metric (Precision@K) as W04 baseline.
The baseline is re-computed here in this notebook on the same test set
so the comparison is apples-to-apples - not numbers from a different run.

Baseline rule (W04):
  score = position_score + maturity_score + engagement_score
  (see W04 notebook for full rule definition)

Model: Random Forest, 200 trees, max_depth=6, class_weight=balanced
  class_weight=balanced because only 17.9% of pages are positive -
  without it the model would predict "not worth investigating" for everything
  and still be right 82% of the time. Balanced weighting forces it to learn
  the minority class.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(df_sorted, k, label_col='label'):
    return df_sorted.head(k)[label_col].mean()

# ── Baseline (W04 rule rebuilt on same test split) ──────────────────────────
test_df = X_test.copy()
test_df['label'] = y_test.values

def baseline_score(row):
    score = 0
    pos = row['avg_position_clean']
    if pd.notna(pos):
        if pos <= 10:   score += 3
        elif pos <= 20: score += 2
        elif pos <= 50: score += 1
    if row['days_with_impressions'] >= 25: score += 2
    elif row['days_with_impressions'] >= 15: score += 1
    eng = row['engagement_rate']
    if eng < 20 and row['impressions_90d'] > 0: score += 2
    elif eng < 50: score += 1
    return score

test_df['baseline_score'] = test_df.apply(baseline_score, axis=1)
base_sorted = test_df.sort_values('baseline_score', ascending=False)

baseline_p20  = precision_at_k(base_sorted, 20)
baseline_p50  = precision_at_k(base_sorted, 50)
baseline_p100 = precision_at_k(base_sorted, 100)

# ── Random Forest ────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

test_df['rf_score'] = rf_probs
rf_sorted = test_df.sort_values('rf_score', ascending=False)

rf_p20  = precision_at_k(rf_sorted, 20)
rf_p50  = precision_at_k(rf_sorted, 50)
rf_p100 = precision_at_k(rf_sorted, 100)

# ── Comparison table ─────────────────────────────────────────────────────────
print(f"{'='*56}")
print(f"{'Method':<26} {'P@20':>8} {'P@50':>8} {'P@100':>8}")
print(f"{'-'*56}")
print(f"{'Base rate (random)':<26} {y_test.mean():>8.3f} {y_test.mean():>8.3f} {y_test.mean():>8.3f}")
print(f"{'Baseline (W04 rule)':<26} {baseline_p20:>8.3f} {baseline_p50:>8.3f} {baseline_p100:>8.3f}")
print(f"{'Random Forest':<26} {rf_p20:>8.3f} {rf_p50:>8.3f} {rf_p100:>8.3f}")
print(f"{'='*56}")
print(f"\nRF beats baseline at P@20 by {rf_p20 - baseline_p20:+.3f}")
print(f"RF beats baseline at P@50 by {rf_p50 - baseline_p50:+.3f}")

Method                         P@20     P@50    P@100
--------------------------------------------------------
Base rate (random)            0.169    0.169    0.169
Baseline (W04 rule)           0.150    0.260    0.240
Random Forest                 0.650    0.540    0.530

RF beats baseline at P@20 by +0.500
RF beats baseline at P@50 by +0.280


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
What the model leans on:

Gini importance (from the tree splits):
  content_age_days      0.412  ← biggest split driver
  impressions_90d       0.244
  days_with_impressions 0.135

Permutation importance (more honest - shuffles one column, measures score drop):
  impressions_90d       0.121  ← actually the most predictive when shuffled
  content_age_days      0.049
  everything else       ~0.000

Interpretation: the model learned to combine age + impressions volume +
days active to find declining pages. This makes intuitive sense -
a page that has been around a long time (age), gets seen a lot (impressions),
but is declining is the core of what the proxy label captures.

The near-zero permutation importance on most features means the model
leans heavily on two signals. This is not a bad thing - it means the
signal is real and concentrated. It also means we could build an almost-
equivalent rule with just two features.

Leakage check on top feature: content_age_days is a static attribute
(how old is this page). It does NOT derive from the label (trend_direction).
impressions_90d is a past measurement. Neither is leaky. ✓

Error analysis: 3 concrete cases
  False positive (model says investigate, label says no):
    Page with 2,426 impressions, position 30, 88 days active, 300 days old.
    Why wrong: page is not declining (trend_direction != down), but the model
    sees old + visible + low engagement and fires. The label requires decline.

  False negative (model missed, label says yes):
    Page with exactly 500 impressions (the minimum floor in our label),
    position 40, declining. The model gave it 0.31 probability.
    Why wrong: it sits right at the boundary of all three label conditions.
    The model is uncertain at the margins - correct behaviour.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# Feature importance - both types
fi_gini = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Gini importance (top 8):")
print(fi_gini.head(8).round(3).to_string())

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
fi_perm = pd.Series(perm.importances_mean, index=features).sort_values(ascending=False)
print("\nPermutation importance (top 8):")
print(fi_perm.head(8).round(4).to_string())

# Error analysis
test_df['rf_pred'] = (rf_probs >= 0.5).astype(int)

print("\n── False positives (model said yes, label said no) ──")
fp = test_df[(test_df['rf_pred']==1) & (test_df['label']==0)].head(3)
print(fp[['impressions_90d','avg_position_clean','days_with_impressions',
          'content_age_days','engagement_rate','rf_score']].round(2).to_string())

print("\n── False negatives (model missed, label said yes) ──")
fn = test_df[(test_df['rf_pred']==0) & (test_df['label']==1)].head(3)
print(fn[['impressions_90d','avg_position_clean','days_with_impressions',
          'content_age_days','engagement_rate','rf_score']].round(2).to_string())

print("\nSummary: model leans on content_age_days + impressions_90d.")
print("False positives are old+visible pages that happen not to be declining.")
print("False negatives are pages sitting right at the boundary of all label conditions.")
print("Neither pattern suggests leakage — they suggest the label is narrow by design.")


Gini importance (top 8):
content_age_days          0.412
impressions_90d           0.244
days_with_impressions     0.135
days_since_last_update    0.061
ctr                       0.037
pageviews_90d             0.026
word_count_filled         0.024
days_with_sessions        0.021

Permutation importance (top 8):
impressions_90d           0.1209
content_age_days          0.0492
days_with_impressions     0.0002
pageviews_90d             0.0001
avg_position_clean        0.0000
days_since_last_update    0.0000
has_position              0.0000
days_with_sessions        0.0000

── False positives (model said yes, label said no) ──
    impressions_90d  avg_position_clean  days_with_impressions  content_age_days  engagement_rate  rf_score
26             2426                30.0                     88               300              0.0      0.84
82             1810                 8.3                     88               348             12.5      0.77
96             1197                21.4    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.